## 1. Chuẩn bị môi trường

In [ ]:
from pathlib import Path
import os
import subprocess

if Path('/kaggle/working').exists():
    PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
    if PROJECT_ROOT.is_dir():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(
            ['git', 'clone', 'https://github.com/tranminhduc9/well-log-imputation.git', str(PROJECT_ROOT)],
            check=True,
        )
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / 'requirements.txt').is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'requirements.txt').is_file():
        raise FileNotFoundError('Open this notebook from the repository root or notebook directory.')
# Must be set before importing torch / initializing CUDA.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(PROJECT_ROOT)
print('Repository:', PROJECT_ROOT)


In [ ]:
%pip install -r requirements.txt matplotlib


## 2. Cấu hình đã chọn

In [ ]:
from argparse import Namespace
import json
import logging
import sys
import torch

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.experiments.geolink import run_experiment

SEEDS = (912, 219)
NEURAL_EPOCHS = 500
NEURAL_PATIENCE = 30
MODEL_SETTINGS = {
    "xgboost": {"max_depth": 3, "learning_rate": 0.03, "n_estimators": 300},
    "brits": {
        "hidden_size": 64, "learning_rate": 0.0005, "batch_size": 32,
        "consistency_weight": 0.1, "weight_decay": 1e-5,
        "mit_weight": 1.0, "mit_reduction": "segment",
        "masking_strategy": "mixed", "gradient_clip": 1.0,
        "validation_metric": "rmse", "min_delta": 0.0,
    },
    "saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "ort_weight": 1.0, "mit_weight": 1.0, "mit_reduction": "segment",
        "min_delta": 0.0,
    },
    # Selected configuration from Table 4.2.
    "conv_saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "encoder_channels": 8, "kernel_size": 15, "conv_expansion": 2,
        "multi_scale_conv": True, "multi_scale_kernels": (3, 7, 15),
        "depth_encoding": False,
        "masking_strategy": "mixed", "mit_reduction": "segment",
        "ort_weight": 1.0, "mit_weight": 1.0, "min_delta": 0.0,
        "masking_rate": 0.2, "independent_shuffle": True,
    },
}
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
if Path("/kaggle/working").exists() and not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Settings before running.")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 3. Huấn luyện và đánh giá

In [ ]:
DATA_ROOT = (Path('/kaggle/input/datasets/minhduc0912/geolink-dataset')
             if Path('/kaggle/working').exists() else PROJECT_ROOT / 'data' / 'processed')
args = Namespace(
    data_root=str(DATA_ROOT), output_dir=None, seeds=SEEDS,
    models=('locf', 'xgboost', 'brits', 'saits', 'conv_saits'),
    evaluate_test=True,
    xgb_estimators=MODEL_SETTINGS['xgboost']['n_estimators'],
    brits_epochs=NEURAL_EPOCHS, brits_patience=NEURAL_PATIENCE,
    saits_epochs=NEURAL_EPOCHS, saits_patience=NEURAL_PATIENCE,
    conv_saits_epochs=NEURAL_EPOCHS, conv_saits_patience=NEURAL_PATIENCE,
)
output_dir = run_experiment(args, PROJECT_ROOT, MODEL_SETTINGS)
print('Results:', output_dir)
summary = pd.read_csv(output_dir / 'summary.csv')
display(summary[['label', 'split', 'scenario', 'n_runs', 'mae_mean', 'mae_std', 'rmse_mean', 'rmse_std']])
history = pd.read_csv(output_dir / 'training_history.csv')
manifest = json.loads((output_dir / 'experiment_results.json').read_text(encoding='utf-8'))


## 4. Hội tụ theo epoch

In [ ]:
for seed in SEEDS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for entry in manifest['runs']:
        if entry['seed'] != seed or entry['model'] in ('locf', 'xgboost'):
            continue
        curve = history[(history['model'] == entry['model']) & (history['seed'] == seed)].sort_values('epoch')
        axes[0].plot(curve['epoch'], curve['loss'], label=entry['label'])
        line, = axes[1].plot(curve['epoch'], curve['validation_rmse'], label=entry['label'])
        best = curve[curve['epoch'] == entry['best_epoch']]
        axes[1].scatter(best['epoch'], best['validation_rmse'], color=line.get_color())
    axes[0].set(title=f'Train loss / seed {seed}', xlabel='Epoch', ylabel='Loss')
    axes[1].set(title=f'Validation RMSE / seed {seed}', xlabel='Epoch', ylabel='Normalized RMSE')
    for axis in axes:
        axis.legend(fontsize=8)
        axis.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_dir / f'convergence_seed_{seed}.png', dpi=150)
    plt.show()

xgb_history = history[history['model'] == 'xgboost']
if not xgb_history.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    logs = json.loads((output_dir / 'preprocessing_parameters.json').read_text(encoding='utf-8'))['log_columns']
    for (seed, log_index), curve in xgb_history.groupby(['seed', 'log_index']):
        curve = curve.sort_values('iteration')
        ax.plot(curve['iteration'], curve['validation_rmse'], label=f'{logs[int(log_index)]} / seed {int(seed)}')
    ax.set(title='XGBoost: Block-20 validation', xlabel='Trees', ylabel='Normalized RMSE')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / 'xgboost_convergence.png', dpi=150)
    plt.show()


## 5. Parameters và thời gian chạy

In [ ]:
import gc
from src.data.loader import DataLoader as WellLogDataLoader
from src.experiments.inference import restore_run
from src.models._training import synchronized_time

WARMUP, REPEATS, N_SAMPLES = 3, 5, 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
preprocessing = json.loads((output_dir / 'preprocessing_parameters.json').read_text(encoding='utf-8'))
test_samples = {
    scenario: WellLogDataLoader(DATA_ROOT, split='test', scenario=scenario).load()['X'][:N_SAMPLES]
    for scenario in preprocessing['missing_scenarios']
}
timings, inference_rows = [], []
for entry in manifest['runs']:
    name, seed = entry['model'], entry['seed']
    model = restore_run({**entry, 'artifact': str(output_dir / entry['artifact'])}, device=DEVICE, batch_size=32)
    device = 'cpu' if name == 'locf' else DEVICE
    network = getattr(model.backend, 'network', None)
    parameters = sum(p.numel() for p in network.parameters()) if network is not None else (0 if name == 'locf' else None)
    curve = history[(history['model'] == name) & (history['seed'] == seed)]
    for scenario, samples in test_samples.items():
        elapsed = []
        for sample in samples:
            inputs = {'X': sample[None]}
            for _ in range(WARMUP):
                model.impute(inputs)
            for _ in range(REPEATS):
                start = synchronized_time(device)
                prediction = model.impute(inputs)
                elapsed.append(1000 * (synchronized_time(device) - start))
                if not np.isfinite(prediction).all():
                    raise ValueError('Non-finite inference output')
        inference_rows.append({'model': name, 'seed': seed, 'scenario': scenario,
                               'samples': len(samples), 'inference_ms_per_sample': np.mean(elapsed)})
    timings.append({
        'model': name, 'seed': seed, 'parameters': parameters,
        'best_epoch': entry['best_epoch'],
        'epochs_run': len(curve) if network is not None else None,
        'train_s_per_epoch': curve['train_seconds'].mean() if network is not None else None,
        'validation_s_per_epoch': curve['validation_seconds'].mean() if network is not None else None,
        'fit_seconds': entry.get('fit_seconds'),
    })
    del model, network
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

inference = pd.DataFrame(inference_rows)
inference.to_csv(output_dir / 'inference_timings.csv', index=False)
latency = inference.groupby(['model', 'seed'], as_index=False)['inference_ms_per_sample'].mean()
timings = pd.DataFrame(timings).merge(latency, on=['model', 'seed'])
timings.to_csv(output_dir / 'runtime_by_seed.csv', index=False)
display(timings)
report = timings.groupby('model', sort=False).mean(numeric_only=True).drop(columns='seed').reset_index()
report.to_csv(output_dir / 'runtime_summary.csv', index=False)
display(report)
history.loc[history['model'].isin(['brits', 'saits', 'conv_saits']),
            ['model', 'seed', 'epoch', 'train_seconds', 'validation_seconds', 'epoch_seconds']].to_csv(
    output_dir / 'epoch_timings.csv', index=False)
print('Inference device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')


## 6. Tải checkpoint, kết quả và biểu đồ

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive(str(output_dir), 'zip', output_dir)
display(FileLink(archive))
